# Visualizações Exploratórias

## Preparação

In [ ]:
# ------------------------------------------------------------------
# PREPARAÇÃO: colunas derivadas que realmente NÃO existem no dataset
# ------------------------------------------------------------------
# Colunas já presentes no CSV que serão REUSADAS:
#   deaths_per_100k_inhabitants      → óbitos acumulados / 100k hab.
#   totalCases_per_100k_inhabitants  → casos acumulados  / 100k hab.
#   deaths_by_totalCases             → razão óbitos/casos (0–1)
#   vaccinated_per_100_inhabitants   → % 1ª dose
#   vaccinated_second_per_100_inhabitants → % 2ª dose
#
# Coluna que PRECISA ser criada:
#   taxa_letalidade → deaths_by_totalCases × 100  (em %)

sub_estados = sub[sub['state'] != 'TOTAL'].copy()

# Converte a razão (0–1) para porcentagem (0–100)
sub_estados['taxa_letalidade'] = sub_estados['deaths_by_totalCases'] * 100

print("Colunas utilizadas nas visualizações:")
cols_viz = [
    'state', 'date', 'region',
    'newCases', 'newDeaths', 'newCases_mm7', 'newDeaths_mm7',
    'deaths_per_100k_inhabitants',
    'totalCases_per_100k_inhabitants',
    'vaccinated_per_100_inhabitants',
    'vaccinated_second_per_100_inhabitants',
    'deaths_by_totalCases',
    'taxa_letalidade',
]
print(sub_estados[cols_viz].describe())


## 1 — Série Temporal: Casos e Óbitos por Estado

In [ ]:
def plot_serie_temporal_casos_obitos(df, estados=None, altura=650):
    """
    Série temporal (linhas) de novos casos e novos óbitos por estado,
    usando médias móveis de 7 dias para suavizar ruídos.

    Parâmetros
    ----------
    df      : DataFrame com 'state', 'date', 'newCases_mm7', 'newDeaths_mm7'
    estados : lista de siglas; se None, usa os 5 maiores em casos totais
    altura  : altura do gráfico em pixels
    """
    if estados is None:
        estados = (
            df.groupby('state')['totalCases'].max()
            .nlargest(5).index.tolist()
        )

    df_plot = df[df['state'].isin(estados)].sort_values('date')
    palette = px.colors.qualitative.Plotly

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=(
            '<b>Novos Casos</b> — Média Móvel 7 dias',
            '<b>Novos Óbitos</b> — Média Móvel 7 dias'
        ),
        vertical_spacing=0.10
    )

    for i, estado in enumerate(estados):
        df_e = df_plot[df_plot['state'] == estado]
        cor  = palette[i % len(palette)]

        fig.add_trace(
            go.Scatter(
                x=df_e['date'], y=df_e['newCases_mm7'],
                name=estado, mode='lines',
                line=dict(color=cor, width=2),
                legendgroup=estado,
            ),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=df_e['date'], y=df_e['newDeaths_mm7'],
                name=estado, mode='lines',
                line=dict(color=cor, width=2, dash='dot'),
                legendgroup=estado,
                showlegend=False,
            ),
            row=2, col=1
        )

    fig.update_layout(
        title='<b>Série Temporal — Novos Casos e Óbitos por Estado</b>',
        title_font_size=18,
        hovermode='x unified',
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        height=altura,
    )
    fig.update_xaxes(title_text='Data', row=2, col=1)
    fig.update_yaxes(title_text='Novos Casos (MM7)',  row=1, col=1)
    fig.update_yaxes(title_text='Novos Óbitos (MM7)', row=2, col=1)
    fig.show()


plot_serie_temporal_casos_obitos(sub_estados)


## 2 — Série Temporal: Vacinação por Estado

In [ ]:
def plot_serie_temporal_vacinacao(df, estados=None, altura=550):
    """
    Série temporal do avanço da vacinação (% da população com 1ª dose)
    por estado, sobreposta à curva de óbitos (MM7) para evidenciar correlação.

    Usa a coluna já existente 'vaccinated_per_100_inhabitants' (porcentagem).

    Parâmetros
    ----------
    df      : DataFrame com 'state', 'date', 'vaccinated_per_100_inhabitants', 'newDeaths_mm7'
    estados : lista de siglas; se None, usa os 5 com maior cobertura vacinal final
    altura  : altura do gráfico em pixels
    """
    if estados is None:
        estados = (
            df.groupby('state')['vaccinated_per_100_inhabitants'].max()
            .nlargest(5).index.tolist()
        )

    df_plot  = df[df['state'].isin(estados)].sort_values('date')
    palette  = px.colors.qualitative.Safe

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    for i, estado in enumerate(estados):
        df_e = df_plot[df_plot['state'] == estado]
        cor  = palette[i % len(palette)]

        # Vacinação — 1ª dose (%) — eixo primário
        fig.add_trace(
            go.Scatter(
                x=df_e['date'], y=df_e['vaccinated_per_100_inhabitants'],
                name=f'{estado} — 1ª Dose (%)',
                mode='lines',
                line=dict(color=cor, width=2),
                legendgroup=estado,
            ),
            secondary_y=False
        )
        # Óbitos MM7 — eixo secundário, tracejado
        fig.add_trace(
            go.Scatter(
                x=df_e['date'], y=df_e['newDeaths_mm7'],
                name=f'{estado} — Óbitos',
                mode='lines',
                line=dict(color=cor, width=1.5, dash='dash'),
                legendgroup=estado,
                showlegend=False,
                opacity=0.6,
            ),
            secondary_y=True
        )

    fig.update_layout(
        title='<b>Série Temporal — Vacinação (1ª Dose %) e Óbitos por Estado</b>',
        title_font_size=18,
        hovermode='x unified',
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        height=altura,
    )
    fig.update_xaxes(title_text='Data')
    fig.update_yaxes(title_text='<b>1ª Dose (% pop.)</b>',    color='SteelBlue', secondary_y=False)
    fig.update_yaxes(title_text='<b>Novos Óbitos (MM7)</b>',  color='Crimson',   secondary_y=True)
    fig.show()


plot_serie_temporal_vacinacao(sub_estados)


## 3 — Mapa Coroplético: Mortalidade por Estado

In [ ]:
def plot_mapa_coropletico(df, altura=600):
    """
    Mapa coroplético do Brasil colorindo cada estado pela taxa de óbitos
    por 100 mil habitantes.

    Usa a coluna já existente 'deaths_per_100k_inhabitants'.

    Parâmetros
    ----------
    df    : DataFrame com 'state', 'deaths_per_100k_inhabitants'
    altura: altura do mapa em pixels
    """
    # Snapshot: valor máximo acumulado por estado
    df_mapa = (
        df.groupby('state')['deaths_per_100k_inhabitants']
        .max()
        .reset_index()
    )

    fig = px.choropleth(
        df_mapa,
        geojson='https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson',
        locations='state',
        featureidkey='properties.sigla',
        color='deaths_per_100k_inhabitants',
        color_continuous_scale='Reds',
        range_color=(
            df_mapa['deaths_per_100k_inhabitants'].min(),
            df_mapa['deaths_per_100k_inhabitants'].max()
        ),
        labels={'deaths_per_100k_inhabitants': 'Óbitos / 100k hab.'},
        title='<b>Mortalidade por COVID-19 — Óbitos por 100 mil Habitantes</b>',
    )

    fig.update_geos(fitbounds='locations', visible=False, bgcolor='rgba(0,0,0,0)')
    fig.update_layout(
        template='plotly_white',
        height=altura,
        coloraxis_colorbar=dict(
            title='Óbitos<br>/ 100k hab.',
            thicknessmode='pixels', thickness=18,
            lenmode='fraction', len=0.75,
        ),
        margin=dict(l=0, r=0, t=60, b=0),
    )
    fig.show()


plot_mapa_coropletico(sub_estados)


## 4 — Boxplot: Mortalidade por Região

In [ ]:
def plot_boxplot_regiao(df, altura=550):
    """
    Boxplot comparando a distribuição de óbitos por 100 mil habitantes
    entre as cinco regiões geográficas do Brasil.

    Usa a coluna já existente 'deaths_per_100k_inhabitants'.

    Parâmetros
    ----------
    df    : DataFrame com 'region', 'deaths_per_100k_inhabitants'
    altura: altura do gráfico em pixels
    """
    df_plot = df.dropna(subset=['deaths_per_100k_inhabitants', 'region'])

    ordem_regioes = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
    palette = {
        'Norte':        '#1f77b4',
        'Nordeste':     '#ff7f0e',
        'Centro-Oeste': '#2ca02c',
        'Sudeste':      '#d62728',
        'Sul':          '#9467bd',
    }

    fig = go.Figure()
    for regiao in ordem_regioes:
        df_r = df_plot[df_plot['region'] == regiao]
        fig.add_trace(
            go.Box(
                y=df_r['deaths_per_100k_inhabitants'],
                name=regiao,
                boxpoints='outliers',
                marker_color=palette.get(regiao, 'gray'),
                line_color=palette.get(regiao, 'gray'),
                fillcolor=palette.get(regiao, 'gray'),
                opacity=0.7,
            )
        )

    fig.update_layout(
        title='<b>Distribuição de Mortalidade por Região</b><br>Óbitos por 100k hab.',
        title_font_size=18,
        yaxis_title='Óbitos por 100k habitantes',
        xaxis_title='Região',
        template='plotly_white',
        showlegend=False,
        height=altura,
    )
    fig.show()


plot_boxplot_regiao(sub_estados)


## 5 — Scatter Plot: Cobertura Vacinal × Mortalidade

In [ ]:
def plot_scatter_vacina_obito(df, altura=600):
    """
    Scatter plot explorando a correlação entre cobertura da 2ª dose (%) e
    óbitos acumulados por 100k hab., com linha de tendência OLS,
    colorido por região e dimensionado pelo total de casos.

    Usa as colunas já existentes:
        'vaccinated_second_per_100_inhabitants' (% 2ª dose)
        'deaths_per_100k_inhabitants'

    Parâmetros
    ----------
    df    : DataFrame — usa snapshot da última data por estado
    altura: altura do gráfico em pixels
    """
    df_snap = (
        df.sort_values('date')
        .groupby('state')
        .last()
        .reset_index()
        .dropna(subset=['vaccinated_second_per_100_inhabitants', 'deaths_per_100k_inhabitants'])
    )

    # Remove zeros que distorceriam a correlação (estados sem dado de vacina)
    df_snap = df_snap[
        (df_snap['vaccinated_second_per_100_inhabitants'] > 0) &
        (df_snap['deaths_per_100k_inhabitants'] > 0)
    ]

    r, p = stats.pearsonr(
        df_snap['vaccinated_second_per_100_inhabitants'],
        df_snap['deaths_per_100k_inhabitants']
    )

    # Linha de tendência manual com numpy (sem statsmodels)
    x_vals = df_snap['vaccinated_second_per_100_inhabitants'].values
    y_vals = df_snap['deaths_per_100k_inhabitants'].values
    coef   = np.polyfit(x_vals, y_vals, 1)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
    y_line = np.polyval(coef, x_line)

    fig = px.scatter(
        df_snap,
        x='vaccinated_second_per_100_inhabitants',
        y='deaths_per_100k_inhabitants',
        color='region',
        size='totalCases',
        text='state',
        labels={
            'vaccinated_second_per_100_inhabitants': '2ª Dose (% da população)',
            'deaths_per_100k_inhabitants':           'Óbitos por 100k hab.',
            'region':                                'Região',
        },
        title=(
            f'<b>Cobertura Vacinal (2ª Dose) × Mortalidade por Estado</b>'
            f'<br>r de Pearson = {r:.3f}  |  p-valor = {p:.4f}'
        ),
        color_discrete_sequence=px.colors.qualitative.Safe,
        template='plotly_white',
        height=altura,
    )
    fig.update_traces(
        textposition='top center',
        selector=dict(mode='markers+text')
    )
    # Adiciona a reta de tendência manualmente
    fig.add_trace(
        go.Scatter(
            x=x_line, y=y_line,
            mode='lines',
            name=f'Tendência (y = {coef[0]:.2f}x + {coef[1]:.1f})',
            line=dict(color='black', width=2, dash='dash'),
            showlegend=True,
        )
    )
    fig.update_layout(
        title_font_size=17,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    )
    fig.show()


plot_scatter_vacina_obito(sub_estados)


## 6 — Heatmap de Correlação: Todas as Variáveis Numéricas

In [ ]:
def plot_heatmap_correlacao(df, altura=800):
    """
    Heatmap da matriz de correlação de Pearson entre todas as variáveis
    numéricas, útil para detectar multicolinearidade antes da modelagem.
    Exibe apenas o triângulo inferior para evitar redundância.

    Parâmetros
    ----------
    df    : DataFrame completo (sub_estados)
    altura: altura do gráfico em pixels
    """
    # Exclui colunas puramente identificadoras ou derivadas de índice
    colunas_excluir = ['epi_week']
    numericas = df.select_dtypes(include='number').drop(
        columns=[c for c in colunas_excluir if c in df.columns],
        errors='ignore'
    )

    corr = numericas.corr(method='pearson').round(2)

    # Máscara triangular superior
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    corr_lower = corr.where(~mask)

    fig = go.Figure(
        go.Heatmap(
            z=corr_lower.values,
            x=corr_lower.columns.tolist(),
            y=corr_lower.index.tolist(),
            colorscale='RdBu_r',
            zmid=0, zmin=-1, zmax=1,
            text=corr_lower.round(2).values,
            texttemplate='%{text}',
            textfont=dict(size=8),
            colorbar=dict(title='r de<br>Pearson'),
            hoverongaps=False,
        )
    )
    fig.update_layout(
        title='<b>Heatmap de Correlação — Variáveis Numéricas</b>',
        title_font_size=18,
        template='plotly_white',
        height=altura,
        xaxis=dict(tickangle=-45),
        margin=dict(l=180, b=180),
    )
    fig.show()


plot_heatmap_correlacao(sub_estados)


## 7 — Gráfico de Barras: Ranking de Letalidade por Estado

In [ ]:
def plot_barras_letalidade(df, altura=620):
    """
    Gráfico de barras horizontais com o ranking de taxa de letalidade
    por estado, ordenado do maior para o menor.

    Usa a coluna derivada 'taxa_letalidade' (deaths_by_totalCases × 100),
    calculada na célula de preparação.

    Parâmetros
    ----------
    df    : DataFrame com 'state', 'taxa_letalidade', 'region'
    altura: altura do gráfico em pixels
    """
    # Snapshot: última data disponível por estado
    df_snap = (
        df.sort_values('date')
        .groupby('state')
        .last()
        .reset_index()
        .dropna(subset=['taxa_letalidade'])
        .sort_values('taxa_letalidade', ascending=True)  # crescente → barras ordenadas
    )

    mapa_cores = {
        'Norte':        '#1f77b4',
        'Nordeste':     '#ff7f0e',
        'Centro-Oeste': '#2ca02c',
        'Sudeste':      '#d62728',
        'Sul':          '#9467bd',
    }
    cores = df_snap['region'].map(mapa_cores).fillna('#aec7e8').tolist()

    fig = go.Figure(
        go.Bar(
            x=df_snap['taxa_letalidade'],
            y=df_snap['state'],
            orientation='h',
            marker_color=cores,
            text=df_snap['taxa_letalidade'].map('{:.2f}%'.format),
            textposition='outside',
            hovertemplate='%{y}: %{x:.2f}%<extra></extra>',
        )
    )

    media = df_snap['taxa_letalidade'].mean()
    fig.add_vline(
        x=media, line_dash='dash', line_color='black',
        annotation_text=f'Média: {media:.2f}%',
        annotation_position='top right',
    )

    # Legenda manual por região
    for regiao, cor in mapa_cores.items():
        fig.add_trace(
            go.Bar(x=[None], y=[None], marker_color=cor, name=regiao, showlegend=True)
        )

    fig.update_layout(
        title='<b>Ranking de Letalidade por Estado</b><br>Taxa = (Óbitos Totais / Casos Totais) × 100',
        title_font_size=18,
        xaxis_title='Taxa de Letalidade (%)',
        yaxis_title='Estado',
        template='plotly_white',
        height=altura,
        margin=dict(r=90),
        barmode='overlay',
        legend=dict(
            title='Região',
            orientation='v',
            yanchor='bottom', y=0.01,
            xanchor='right',  x=0.99,
        ),
    )
    fig.show()


plot_barras_letalidade(sub_estados)
